# Week 16, Day 1: Conditional probability: joint, marginal and conditional distributions

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w016/d1.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Two conditionals, two answers

Simulate 200,000 trials of the prediction's experiment. Estimate `p_burst_given_pref` and `p_pref_given_burst` by counting, and compute `exact_value` for the second from the definition of conditional probability.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(2004)
T = 200000
pref = rng.random(T) < 0.02                                   # was the preferred stimulus shown?
burst = np.where(pref, rng.random(T) < 0.9, rng.random(T) < 0.05)

p_burst_given_pref = ...          # among trials with the preferred stimulus, the fraction with a burst
p_pref_given_burst = ...          # among trials with a burst, the fraction with the preferred stimulus
exact_value = ...                 # P(pref and burst) / P(burst)
print(f"P(burst | preferred) = {p_burst_given_pref:.3f};  P(preferred | burst) = {p_pref_given_burst:.3f} (exact {exact_value:.3f})")

In [ ]:
import numpy as np
def test_forward():
    "P(burst | preferred) is close to 0.9 in the simulation"
    assert abs(p_burst_given_pref - 0.9) < 0.02

def test_backward():
    "P(preferred | burst) matches the exact value 0.9 * 0.02 / (0.9 * 0.02 + 0.05 * 0.98) to within 0.02"
    exact = 0.9 * 0.02 / (0.9 * 0.02 + 0.05 * 0.98)
    assert abs(p_pref_given_burst - exact) < 0.02 and abs(exact_value - exact) < 1e-12

test_forward()
test_backward()
print("All checks passed")

### Exercise: Conditional distributions from a joint table

Two neurons share a fluctuating gain. Build their joint table of counts, the conditional distributions `cond[a, b]` = P(B = b | A = a), and the conditional means `cond_mean[a]` = E[B | A = a].

In [ ]:
T = 100000
g = rng.gamma(4.0, 1 / 4.0, T)                     # shared gain, mean 1
a, b = rng.poisson(3 * g), rng.poisson(4 * g)
joint = np.zeros((a.max() + 1, b.max() + 1))
np.add.at(joint, (a, b), 1)
joint /= T

pA = joint.sum(axis=1)
cond = ...                                          # divide each row by its total (leave empty rows as zeros)
cond_mean = ...
print("E[B] =", round(b.mean(), 3)); print("E[B | A = a] for a = 0..8:", cond_mean[:9].round(2))
plt.plot(np.arange(9), cond_mean[:9], "o-"); plt.axhline(b.mean(), ls="--", c="k"); plt.xlabel("neuron A count"); plt.ylabel("E[B | A]"); plt.show()

In [ ]:
import numpy as np
def test_rows():
    "each row of cond sums to 1 where neuron A's count was observed"
    ok = joint.sum(1) > 0
    assert np.allclose(cond[ok].sum(1), 1)

def test_cond_mean():
    "the conditional mean of B rises with A's count over counts 0 to 8"
    assert np.all(np.diff(cond_mean[:9]) > 0)

test_rows()
test_cond_mean()
print("All checks passed")

### Exercise: Split the variance

Check the law of total probability by rebuilding P(B) from `cond` and `pA` (store it in `pB_total`). Then split Var(B) using the gain: given g, B is Poisson with mean 4g, so Var(B | g) = 4g and E[B | g] = 4g. Store `within` = E[Var(B | g)] and `between` = Var(E[B | g]).

In [ ]:
pB_total = ...
within = ...
between = ...
print(f"Var(B) = {np.var(b):.3f};  within (Poisson) {within:.3f} + between (gain) {between:.3f} = {within + between:.3f}")
print(f"fraction of B's variance explained by the gain: {between / np.var(b):.2f}")

In [ ]:
import numpy as np
def test_total_prob():
    "the law of total probability reproduces the marginal of B"
    assert np.allclose(pB_total, joint.sum(0))

def test_total_var():
    "the two parts of the variance add up to Var(B) to within 1%"
    assert abs(within + between - np.var(b)) / np.var(b) < 0.01

def test_gain_part():
    "the gain explains about half of B's variance here (between 40% and 60%)"
    assert 0.4 < between / np.var(b) < 0.6

test_total_prob()
test_total_var()
test_gain_part()
print("All checks passed")

### Exercise: Correlated overall, independent given the gain

Compute the overall correlation `r_overall` between a and b. Then split the trials into 20 bins of equal size by the value of g, compute the correlation within each bin, and store the average in `r_within`.

In [ ]:
r_overall = ...
order = np.argsort(g)
bins = np.array_split(order, 20)
r_within = ...
print(f"correlation overall {r_overall:.3f};  average correlation within gain bins {r_within:.3f}")

In [ ]:
import numpy as np
def test_overall():
    "overall, A and B are clearly correlated (r above 0.3)"
    assert r_overall > 0.3

def test_within():
    "within narrow bins of the gain, the average correlation is near 0 (below 0.03 in size)"
    assert abs(r_within) < 0.03

test_overall()
test_within()
print("All checks passed")